#Paquetes necesarios

In [2]:
import cv2  
import math 

from ultralytics import YOLO

Modelos preentrenados, visualizando con las utilidades de ultralytics

In [3]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 119.4ms
video 1/1 (frame 2/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 92.0ms
video 1/1 (frame 3/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 90.2ms
video 1/1 (frame 4/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C00

In [ ]:

from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]


# Captura desde la webcam
vid = cv2.VideoCapture("C0142.MP4")
track_history = defaultdict(lambda: [])

# Para guardar el vídeo procesado para ello necesitamos el códec, los fps y el tamaño del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Guardar el vídeo procesado
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}
  
while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2) # convert to int values

                #Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

                if box.id is not None:
                    nombre_clase = classNames[cls]
                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        out.write(img)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
out.release()
# Destruye ventanas
cv2.destroyAllWindows()



0: 384x640 1 car, 83.6ms
Speed: 4.3ms preprocess, 83.6ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.85
Clase --> car

0: 384x640 1 car, 80.1ms
Speed: 6.2ms preprocess, 80.1ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 104.7ms
Speed: 4.6ms preprocess, 104.7ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 104.9ms
Speed: 5.0ms preprocess, 104.9ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.84
Clase --> car

0: 384x640 1 car, 85.6ms
Speed: 4.9ms preprocess, 85.6ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 107.7ms
Speed: 6.6ms preprocess, 107.7ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.81
Clase --> car

0: 384x640 1 car, 136.2ms
Speed: 6.5ms preprocess, 

Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [5]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 88.6ms
video 1/1 (frame 2/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 114.3ms
video 1/1 (frame 3/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 87.1ms
video 1/1 (frame 4/375) c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\TGC23_PdH_C0056